In [ ]:
!pip install boto3

In [147]:
import json
import csv
import pandas as pd
import matplotlib.pyplot as plt
import boto3
from io import BytesIO
import numpy as np
from urllib.parse import urlparse
import math
import os

%matplotlib inline

In [57]:
client = boto3.client("s3")

def list_all_objects_in_s3(bucketName, prefixName=None):
    """Get a list of objects in an S3 bucket path.
    @params:
    bucketName : s3 bucket
    prefixName : s3 prefix to bucket
    returns : list of objects in s3 path
    [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}"]
    """
    paginator = client.get_paginator('list_objects_v2')
    if prefixName:
        pages = paginator.paginate(Bucket=bucketName, StartAfter=prefixName)
    else:
        pages = paginator.paginate(Bucket=bucketName)
    list_of_files=[]
    for page in pages:
        for obj in page['Contents']:
            files = bucketName+'/'+obj['Key']
            if prefixName:
                if bucketName+'/'+prefixName in files:
                    if bucketName+'/'+prefixName+'/' !=files:
                        list_of_files.append(files)
            else:
                list_of_files.append(files)
    return list_of_files

def read_json_file_from_s3(bucketName, prefixName, fileName):
    """ returns json file content
        @param
        bucketName : s3 bucket
        prefixName : s3 prefix to bucket
        fileName   : input json file Name
        [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}"]
    """
    result = client.list_objects(Bucket = bucketName, Prefix=prefixName)
    for o in result.get('Contents'):
        data = client.get_object(Bucket=bucketName, Key=o.get('Key'))
        contents = data['Body'].read()
    return json.loads(contents.decode("utf-8"))

def coco_to_yolo(x1, y1, w, h, image_w, image_h):
    return [(x1 + w/2)/image_w, (y1 + h/2)/image_h, w/image_w, h/image_h]

def s3_copy(srcbucket, srckey, destbucket, destkey):
    try:
        s3 = boto3.resource('s3')
        copy_source = {
            'Bucket': srcbucket,
            'Key': srckey
        }
        s3.meta.client.copy(copy_source, destbucket, destkey)
    except Exception as e:
        print('s3 copy failed. '+ str(e)+'for file: '+copy_source)
    return 

def to_s3_npy(data: np.array, s3_uri: str):
    # s3_uri looks like f"s3://{BUCKET_NAME}/{KEY}"
    """
    @params 
    data : file content 
    s3_uri : s3 file path along with name [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}/{fileName.npy}"]
    """
    try:
        bytes_ = BytesIO()
        np.save(bytes_, data, allow_pickle=True)
        bytes_.seek(0)
        parsed_s3 = urlparse(s3_uri)
        client.upload_fileobj(
            Fileobj=bytes_, Bucket=parsed_s3.netloc, Key=parsed_s3.path[1:]
        )
    except Exception as e: # work on python 3.x
        print('Failed to write numpy array to S3: '+ str(e))
    return True

def from_s3_npy(s3_uri: str):
    """
    @params 
    data : file content 
    s3_uri : s3 file path along with name [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}/{fileName.npy}"]
    """
    try:
        bytes_ = BytesIO()
        parsed_s3 = urlparse(s3_uri)
        client.download_fileobj(
            Fileobj=bytes_, Bucket=parsed_s3.netloc, Key=parsed_s3.path[1:]
        )
        bytes_.seek(0)
        np_data=np.load(bytes_, allow_pickle=True)
    
    except Exception as e: # work on python 3.x
        print('Failed to read numpy array from S3: '+ str(e))
    return np_data

In [58]:
list_all_objects_in_s3("skm-dataset", "skm-tea/qdess/v1-release/annotations/v1.0.0")

['skm-dataset/skm-tea/qdess/v1-release/annotations/v1.0.0/test.json',
 'skm-dataset/skm-tea/qdess/v1-release/annotations/v1.0.0/train.json',
 'skm-dataset/skm-tea/qdess/v1-release/annotations/v1.0.0/val.json']

In [59]:
filenames = ["test.json", "train.json", "val.json"]
print("annotation files:", filenames)

annotation files: ['test.json', 'train.json', 'val.json']


In [ ]:
for file in filenames:
    
    bbox = pd.DataFrame()
    merge_annot = pd.DataFrame()

    print("reading file:", file)
    outputFileName = file.split(".")[0] + "_annotations"
    print("writting annotations to file:", outputFileName)
    
    
    f = open(file)
    metadata = json.load(f)
    tissues = pd.DataFrame(metadata["tissues"])
    images = pd.DataFrame(metadata["images"])
    images = images.rename(columns={"id":"image_id"})
    annotations= pd.DataFrame(metadata["annotations"])
    categories = pd.DataFrame(metadata["categories"])
    
    tissue_dict = dict(zip(tissues.id, tissues.name))
    cat_dict = dict(zip(categories.id, categories.name))
    cat_id_to_supercat_id = dict(zip(categories.id, categories.supercategory_id))
    supercat_dict = dict(zip(categories.supercategory_id, categories.supercategory))
    annotations["category"] = annotations["category_id"].map(cat_dict)
    annotations["supercat_id"] = annotations["category_id"].map(cat_id_to_supercat_id) 
    annotations["supercategory"] = annotations["supercat_id"].map(supercat_dict)
    annotations["tissues"] = annotations["tissue_id"].map(tissue_dict)
    
    merge_annot = pd.merge(annotations, images, on="image_id", how="left")
    merge_annot = merge_annot.drop(["timepoint", "voxel_spacing", "matrix_shape", "orientation", "id", "inspected", "num_echoes"], axis=1)
    merge_annot["split"] = file.split(".")[0]

    for ind in merge_annot.index:
        dict_row = merge_annot.iloc[ind].to_dict()
        if not np.isnan(dict_row["bbox"]).any():
            for i in range(int(dict_row["bbox"][2]), (int(dict_row["bbox"][2]) + int(dict_row["bbox"][5]))):
                dict_row["sl"] = i
                dict_row["2d_bbox"] = [dict_row["bbox"][1], dict_row["bbox"][0], dict_row["bbox"][4], dict_row["bbox"][3]]
                dict_row["yolo_bbox"] = coco_to_yolo(dict_row["bbox"][1], dict_row["bbox"][0], dict_row["bbox"][4], dict_row["bbox"][3], 512, 512)
                bbox = pd.concat([bbox, pd.DataFrame([dict_row])], ignore_index=True)
    bbox.to_csv(outputFileName+".csv", index=False)
    print("annotation file created!\n")

In [60]:
train = pd.read_csv("train_annotations.csv")
val = pd.read_csv("val_annotations.csv")
test = pd.read_csv("test_annotations.csv")

In [ ]:
all_annotations = pd.DataFrame()
all_annotations = pd.concat([all_annotations, train, val, test], ignore_index=True)

In [ ]:
all_annotations

In [61]:
test["scan_id"].unique()

array(['MTR_173', 'MTR_176', 'MTR_178', 'MTR_188', 'MTR_196', 'MTR_198',
       'MTR_219', 'MTR_221', 'MTR_218', 'MTR_199', 'MTR_224', 'MTR_227',
       'MTR_235', 'MTR_237', 'MTR_240', 'MTR_241', 'MTR_244', 'MTR_248',
       'MTR_005', 'MTR_006', 'MTR_030', 'MTR_034', 'MTR_048', 'MTR_052',
       'MTR_065', 'MTR_066', 'MTR_079', 'MTR_080', 'MTR_096', 'MTR_099',
       'MTR_120', 'MTR_126', 'MTR_144', 'MTR_156', 'MTR_158'],
      dtype=object)

In [62]:
val["scan_id"].unique()

array(['MTR_133', 'MTR_149', 'MTR_220', 'MTR_037', 'MTR_083', 'MTR_110',
       'MTR_139', 'MTR_182', 'MTR_208', 'MTR_203', 'MTR_047', 'MTR_069',
       'MTR_071', 'MTR_112', 'MTR_154', 'MTR_184', 'MTR_207', 'MTR_222',
       'MTR_023', 'MTR_044', 'MTR_062', 'MTR_115', 'MTR_130', 'MTR_152',
       'MTR_167', 'MTR_026', 'MTR_045'], dtype=object)

In [63]:
train["scan_id"].unique()

array(['MTR_104', 'MTR_105', 'MTR_179', 'MTR_194', 'MTR_046', 'MTR_174',
       'MTR_177', 'MTR_211', 'MTR_212', 'MTR_225', 'MTR_015', 'MTR_159',
       'MTR_018', 'MTR_172', 'MTR_145', 'MTR_057', 'MTR_189', 'MTR_001',
       'MTR_008', 'MTR_016', 'MTR_020', 'MTR_033', 'MTR_040', 'MTR_087',
       'MTR_095', 'MTR_097', 'MTR_113', 'MTR_164', 'MTR_129', 'MTR_193',
       'MTR_210', 'MTR_243', 'MTR_051', 'MTR_053', 'MTR_077', 'MTR_098',
       'MTR_140', 'MTR_142', 'MTR_150', 'MTR_163', 'MTR_183', 'MTR_190',
       'MTR_200', 'MTR_201', 'MTR_204', 'MTR_209', 'MTR_229', 'MTR_232',
       'MTR_251', 'MTR_056', 'MTR_094', 'MTR_100', 'MTR_107', 'MTR_124',
       'MTR_135', 'MTR_161', 'MTR_175', 'MTR_186', 'MTR_230', 'MTR_233',
       'MTR_236', 'MTR_245', 'MTR_019', 'MTR_049', 'MTR_061', 'MTR_084',
       'MTR_093', 'MTR_101'], dtype=object)

In [64]:
test["image"] = test["scan_id"].str.cat(test["sl"].values.astype(str), sep="_")


In [65]:
test[test["scan_id"]=="MTR_248"]

,image_id,category_id,tissue_id,bbox,confidence,labeler,category,supercat_id,supercategory,tissues,file_name,msp_id,msp_file_name,scan_id,subject_id,split,sl,2d_bbox,yolo_bbox,image
2175,36,16,-1,"[1.0, 82.0, 12.0, 327.0, 160.0, 119.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,12,"[82.0, 1.0, 160.0, 327.0]","[0.31640625, 0.3212890625, 0.3125, 0.638671875]",MTR_248_12
2176,36,16,-1,"[1.0, 82.0, 12.0, 327.0, 160.0, 119.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,13,"[82.0, 1.0, 160.0, 327.0]","[0.31640625, 0.3212890625, 0.3125, 0.638671875]",MTR_248_13
2177,36,16,-1,"[1.0, 82.0, 12.0, 327.0, 160.0, 119.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,14,"[82.0, 1.0, 160.0, 327.0]","[0.31640625, 0.3212890625, 0.3125, 0.638671875]",MTR_248_14
2178,36,16,-1,"[1.0, 82.0, 12.0, 327.0, 160.0, 119.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,15,"[82.0, 1.0, 160.0, 327.0]","[0.31640625, 0.3212890625, 0.3125, 0.638671875]",MTR_248_15
2179,36,16,-1,"[1.0, 82.0, 12.0, 327.0, 160.0, 119.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,16,"[82.0, 1.0, 160.0, 327.0]","[0.31640625, 0.3212890625, 0.3125, 0.638671875]",MTR_248_16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2351,36,13,4,"[277.0, 144.0, 85.0, 39.0, 25.0, 22.0]",4.0,1,Cartilage Lesion (2A),3,Cartilage Lesion,Femoral Cartilage,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,102,"[144.0, 277.0, 25.0, 39.0]","[0.3056640625, 0.5791015625, 0.048828125, 0.07...",MTR_248_102
2352,36,13,4,"[277.0, 144.0, 85.0, 39.0, 25.0, 22.0]",4.0,1,Cartilage Lesion (2A),3,Cartilage Lesion,Femoral Cartilage,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,103,"[144.0, 277.0, 25.0, 39.0]","[0.3056640625, 0.5791015625, 0.048828125, 0.07...",MTR_248_103
2353,36,13,4,"[277.0, 144.0, 85.0, 39.0, 25.0, 22.0]",4.0,1,Cartilage Lesion (2A),3,Cartilage Lesion,Femoral Cartilage,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,104,"[144.0, 277.0, 25.0, 39.0]","[0.3056640625, 0.5791015625, 0.048828125, 0.07...",MTR_248_104
2354,36,13,4,"[277.0, 144.0, 85.0, 39.0, 25.0, 22.0]",4.0,1,Cartilage Lesion (2A),3,Cartilage Lesion,Femoral Cartilage,MTR_248.h5,0000124_V00,0000124_V00.h5,MTR_248,124,test,105,"[144.0, 277.0, 25.0, 39.0]","[0.3056640625, 0.5791015625, 0.048828125, 0.07...",MTR_248_105


In [66]:
len(test.image.unique())

3110

In [ ]:
test_image_list = list(test.image)
echo1_files = list_all_objects_in_s3('skm-dataset-echo1')
echo2_files = list_all_objects_in_s3('skm-dataset-echo2')

count = 0

for image in test_image_list:
    if image in echo1_files:
        srcbucket = "skm-dataset-echo1"
    else:
        srcbucket = "skm-dataset-echo2"

    arr = from_s3_npy("s3://" + srcbucket + "/" + image + ".npy")
    arr = np.expand_dims(arr, axis=2)
    arr = np.repeat(arr, 3, axis=2)

    filename = "s3://skm-yolo/test/images/" + image + ".npy"
    to_s3_npy(arr, filename)
    count += 1
    if count % 50 == 0:
        print(count)

In [67]:
len(list_all_objects_in_s3("skm-yolo", "test/images"))

3110

In [68]:
val["image"] = val["scan_id"].str.cat(val["sl"].values.astype(str), sep="_")

In [69]:
len(list(val.image.unique()))

1801

In [ ]:
val_image_list = list(val.image.unique())
echo1_files = list_all_objects_in_s3('skm-dataset-echo1')
echo2_files = list_all_objects_in_s3('skm-dataset-echo2')

for image in val_image_list:
    if image in echo1_files:
        srcbucket = "skm-dataset-echo1"
    else:
        srcbucket = "skm-dataset-echo2"

    arr = from_s3_npy("s3://" + srcbucket + "/" + image + ".npy")
    arr = np.expand_dims(arr, axis=2)
    arr = np.repeat(arr, 3, axis=2)

    filename = "s3://skm-yolo/valid/images" + image + ".npy"
    to_s3_npy(arr, filename)
    
    if len(list_all_objects_in_s3("skm-yolo", "valid/images")) % 20 == 0:
        print(len(list_all_objects_in_s3("skm-yolo", "valid/images")))

In [70]:
len(list_all_objects_in_s3("skm-yolo", "valid/images"))

1801

In [71]:
train["image"] = train["scan_id"].str.cat(train["sl"].values.astype(str), sep="_")

In [72]:
len(list(train.image.unique()))

4469

In [ ]:
train_image_list = list(train.image.unique())
echo1_files = list_all_objects_in_s3('skm-dataset-echo1')
echo2_files = list_all_objects_in_s3('skm-dataset-echo2')

for image in train_image_list:
    if image in echo1_files:
        srcbucket = "skm-dataset-echo1"
    else:
        srcbucket = "skm-dataset-echo2"

    arr = from_s3_npy("s3://" + srcbucket + "/" + image + ".npy")
    arr = np.expand_dims(arr, axis=2)
    arr = np.repeat(arr, 3, axis=2)

    filename = "s3://skm-yolo/train/images/" + image + ".npy"
    to_s3_npy(arr, filename)
    
    if len(list_all_objects_in_s3("skm-yolo", "train/images")) % 50 == 0:
        print(len(list_all_objects_in_s3("skm-yolo", "train/images")))

In [73]:
len(list_all_objects_in_s3("skm-yolo", "train/images"))

4359

In [74]:
train_image_list = list(train.image.unique())
echo1_files = list_all_objects_in_s3('skm-dataset-echo1')
train_s3_list = list_all_objects_in_s3("skm-yolo", "train/images")
train_s3_list = [file.split("/")[-1] for file in train_s3_list]
train_s3_list = [name.split(".")[0] for name in train_s3_list]
#echo2_files = list_all_objects_in_s3('skm-dataset-echo2')

missing_files = set(train_image_list) - set(train_s3_list)

In [75]:
len(missing_files)

110

In [76]:
train = train[train["sl"] <= 160]

In [77]:
train[train["sl"] > 160]

,image_id,category_id,tissue_id,bbox,confidence,labeler,category,supercat_id,supercategory,tissues,file_name,msp_id,msp_file_name,scan_id,subject_id,split,sl,2d_bbox,yolo_bbox,image


In [78]:
train_image_list = list(train.image.unique())
echo1_files = list_all_objects_in_s3('skm-dataset-echo1')
train_s3_list = list_all_objects_in_s3("skm-yolo", "train/images")
train_s3_list = [file.split("/")[-1] for file in train_s3_list]
train_s3_list = [name.split(".")[0] for name in train_s3_list]
#echo2_files = list_all_objects_in_s3('skm-dataset-echo2')

missing_files = set(train_image_list) - set(train_s3_list)

In [79]:
len(missing_files)

0

In [80]:
len(train.image.unique())

4359

In [35]:
for image in missing_files:
    if image in echo1_files:
        srcbucket = "skm-dataset-echo1"
    else:
        srcbucket = "skm-dataset-echo2"
        
    try:
        arr = from_s3_npy("s3://" + srcbucket + "/" + image + ".npy")
        arr = np.expand_dims(arr, axis=2)
        arr = np.repeat(arr, 3, axis=2)

        filename = "s3://skm-yolo/train/images/" + image + ".npy"
        to_s3_npy(arr, filename)
        
    except:
        print(f"{image} error")
        
    if len(list_all_objects_in_s3("skm-yolo", "train/images")) % 50 == 0:
        print(len(list_all_objects_in_s3("skm-yolo", "train/images")))

3550
3600
3650
3700
3750
3800
3850
3900
3950
4000
4050
4100
4150
4200
4250
4300
4350


In [81]:
len(list_all_objects_in_s3("skm-yolo", "train/images"))

4359

In [82]:
test

,image_id,category_id,tissue_id,bbox,confidence,labeler,category,supercat_id,supercategory,tissues,file_name,msp_id,msp_file_name,scan_id,subject_id,split,sl,2d_bbox,yolo_bbox,image
0,18,16,-1,"[109.0, 131.0, 20.0, 167.0, 120.0, 38.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_173.h5,0000186_V00,0000186_V00.h5,MTR_173,186,test,20,"[131.0, 109.0, 120.0, 167.0]","[0.373046875, 0.3759765625, 0.234375, 0.326171...",MTR_173_20
1,18,16,-1,"[109.0, 131.0, 20.0, 167.0, 120.0, 38.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_173.h5,0000186_V00,0000186_V00.h5,MTR_173,186,test,21,"[131.0, 109.0, 120.0, 167.0]","[0.373046875, 0.3759765625, 0.234375, 0.326171...",MTR_173_21
2,18,16,-1,"[109.0, 131.0, 20.0, 167.0, 120.0, 38.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_173.h5,0000186_V00,0000186_V00.h5,MTR_173,186,test,22,"[131.0, 109.0, 120.0, 167.0]","[0.373046875, 0.3759765625, 0.234375, 0.326171...",MTR_173_22
3,18,16,-1,"[109.0, 131.0, 20.0, 167.0, 120.0, 38.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_173.h5,0000186_V00,0000186_V00.h5,MTR_173,186,test,23,"[131.0, 109.0, 120.0, 167.0]","[0.373046875, 0.3759765625, 0.234375, 0.326171...",MTR_173_23
4,18,16,-1,"[109.0, 131.0, 20.0, 167.0, 120.0, 38.0]",5.0,1,Effusion,4,Effusion,NaN,MTR_173.h5,0000186_V00,0000186_V00.h5,MTR_173,186,test,24,"[131.0, 109.0, 120.0, 167.0]","[0.373046875, 0.3759765625, 0.234375, 0.326171...",MTR_173_24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5291,17,16,-1,"[34.0, 111.0, 65.0, 244.0, 139.0, 77.0]",4.0,2,Effusion,4,Effusion,NaN,MTR_158.h5,0000169_V00,0000169_V00.h5,MTR_158,169,test,137,"[111.0, 34.0, 139.0, 244.0]","[0.3525390625, 0.3046875, 0.271484375, 0.4765625]",MTR_158_137
5292,17,16,-1,"[34.0, 111.0, 65.0, 244.0, 139.0, 77.0]",4.0,2,Effusion,4,Effusion,NaN,MTR_158.h5,0000169_V00,0000169_V00.h5,MTR_158,169,test,138,"[111.0, 34.0, 139.0, 244.0]","[0.3525390625, 0.3046875, 0.271484375, 0.4765625]",MTR_158_138
5293,17,16,-1,"[34.0, 111.0, 65.0, 244.0, 139.0, 77.0]",4.0,2,Effusion,4,Effusion,NaN,MTR_158.h5,0000169_V00,0000169_V00.h5,MTR_158,169,test,139,"[111.0, 34.0, 139.0, 244.0]","[0.3525390625, 0.3046875, 0.271484375, 0.4765625]",MTR_158_139
5294,17,16,-1,"[34.0, 111.0, 65.0, 244.0, 139.0, 77.0]",4.0,2,Effusion,4,Effusion,NaN,MTR_158.h5,0000169_V00,0000169_V00.h5,MTR_158,169,test,140,"[111.0, 34.0, 139.0, 244.0]","[0.3525390625, 0.3046875, 0.271484375, 0.4765625]",MTR_158_140


In [83]:
test["supercat_id"] = test["supercat_id"] - 1

In [153]:
# TEST CASE

for ind in a.index:
    dict_row = a.iloc[ind].to_dict()
    dict_row["yolo_bbox"] = [float(x.strip(" []")) for x in dict_row["yolo_bbox"].split(",")]
    new_dict_row = {}
    df = pd.DataFrame()
    if not np.isnan(dict_row["yolo_bbox"]).any():
        new_dict_row["class"] = dict_row["supercat_id"]
        new_dict_row["yolo_cx"] = dict_row["yolo_bbox"][0]
        new_dict_row["yolo_cy"] = dict_row["yolo_bbox"][1]
        new_dict_row["yolo_w"] = dict_row["yolo_bbox"][2]
        new_dict_row["yolo_h"] = dict_row["yolo_bbox"][3]
        df = pd.concat([df, pd.DataFrame([new_dict_row])], ignore_index=True)
        outputFileName = dict_row["image"]
        if os.path.isfile(outputFileName+".txt"):
            with open(outputFileName+".txt", "a") as f:
                df.to_csv(f, index=False, header=None)
        else:
            df.to_csv(outputFileName+".txt", index=False, header=None)


In [280]:
def write_file_to_s3(bucketName, fileContent, fileName, fileType="dataframe", prefixName=None):
    try:
        if prefixName:
            fileName=prefixName+'/'+fileName
        if fileType=='dataframe':
            fileContent.to_csv('s3://'+bucketName+'/'+fileName+'.txt', index=False, header=None)
        else:
            response = client.put_object(Bucket=bucketName, #'testshi'
                                         Body=fileContent, # '{"content": "bytes or seekable file-like object"}',
                                         Key=fileName) #'test.json' # 'Object key for which the PUT operation was initiated'
        print('Writing to S3 completed for ',fileName)
    except Exception as e: # work on python 3.x
        print('Failed to write numpy to S3: '+ str(e))
    return

s3 = boto3.resource('s3')
def read_file_from_s3(bucket, file):
    try:
        obj = s3.Object(bucket, file)
        data = obj.get()["Body"]
        
    except Exception as e: # work on python 3.x
        print('Failed to read file from S3: '+ str(e))
    return data.read().decode(encoding='utf-8')

def set_key(dictionary, key, value):
    if type(dictionary[key]) == list:
        dictionary[key].append(value)
    else:
        dictionary[key] = [dictionary[key], value]

In [283]:
len(test.image.unique())

3110

In [306]:
test_df = test

In [316]:
for ind in test_df.index:
    dict_row = test_df.iloc[ind].to_dict()
    dict_row["yolo_bbox"] = [float(x.strip(" []")) for x in dict_row["yolo_bbox"].split(",")]
    new_dict_row = {}
    df = pd.DataFrame()
    file_list = [file.split("/")[-1] for file in list_all_objects_in_s3("skm-yolo", "test/labels")]
    if not np.isnan(dict_row["yolo_bbox"]).any():
        new_dict_row["class"] = dict_row["supercat_id"]
        new_dict_row["yolo_cx"] = dict_row["yolo_bbox"][0]
        new_dict_row["yolo_cy"] = dict_row["yolo_bbox"][1]
        new_dict_row["yolo_w"] = dict_row["yolo_bbox"][2]
        new_dict_row["yolo_h"] = dict_row["yolo_bbox"][3]        
        outputFileName = dict_row["image"]
        if (outputFileName+".txt") in file_list:
            content = read_file_from_s3("skm-yolo", "test/labels/"+outputFileName+".txt").strip("\n")
            for sub_content in content.split("\n"):
                set_key(new_dict_row, "class", int(sub_content.split(",")[0]))
                set_key(new_dict_row, "yolo_cx", float(sub_content.split(",")[1]))
                set_key(new_dict_row, "yolo_cy", float(sub_content.split(",")[2]))
                set_key(new_dict_row, "yolo_w", float(sub_content.split(",")[3]))
                set_key(new_dict_row, "yolo_h", float(sub_content.split(",")[4]))
            df = pd.concat([df, pd.DataFrame(new_dict_row)], ignore_index=True)
            write_file_to_s3("skm-yolo", df, outputFileName, "dataframe", "test/labels")
            
        else:
            df = pd.concat([df, pd.DataFrame([new_dict_row])], ignore_index=True)
            write_file_to_s3("skm-yolo", df, outputFileName, "dataframe", "test/labels")
            
        if len(list_all_objects_in_s3("skm-yolo", "test/labels")) % 20 == 0:
            print(len(list_all_objects_in_s3("skm-yolo", "test/labels")))

Writing to S3 completed for  test/labels/MTR_173_21
Writing to S3 completed for  test/labels/MTR_173_23
Writing to S3 completed for  test/labels/MTR_173_25
Writing to S3 completed for  test/labels/MTR_173_27
Writing to S3 completed for  test/labels/MTR_173_29
Writing to S3 completed for  test/labels/MTR_173_31
Writing to S3 completed for  test/labels/MTR_173_33
Writing to S3 completed for  test/labels/MTR_173_35
Writing to S3 completed for  test/labels/MTR_173_37
Writing to S3 completed for  test/labels/MTR_173_39
Writing to S3 completed for  test/labels/MTR_173_41
Writing to S3 completed for  test/labels/MTR_173_43
1980
Writing to S3 completed for  test/labels/MTR_173_45
Writing to S3 completed for  test/labels/MTR_173_47
Writing to S3 completed for  test/labels/MTR_173_49
Writing to S3 completed for  test/labels/MTR_173_51
Writing to S3 completed for  test/labels/MTR_173_53
Writing to S3 completed for  test/labels/MTR_173_55
Writing to S3 completed for  test/labels/MTR_173_57
Writing

In [317]:
print("done")

done


In [318]:
len(list_all_objects_in_s3("skm-yolo", "test/labels/"))

3110